# Let's change our camera's background in a notebook

Shoutout to [MODNet](https://github.com/ZHKKKe/MODNet) and the
[ExecuTorch background-removal demo](https://executorch.ai/playground/webgpu/#/demo/bgremove) for the inspiration!

Try the included portrait or turn on your camera. Remove the background,
blur it, change its colour, or view the foreground mask.
Your photos and camera frames stay in this browser.

Choose CPU or WebGPU with `DEVICE` and run the cells from top to bottom in a fresh kernel.


In [ ]:
import base64
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from IPython.display import display
from modnet_models.modnet import MODNet

DEVICE = 'webgpu'
SIZE = 256
torch.autograd.set_multithreading_enabled(False)
print(f'PyTorch {torch.__version__} | OpenCV {cv2.__version__} | {DEVICE}')


## 1. Load the pretrained model

MODNet predicts a soft foreground mask around a person, including partially
transparent edges such as hair. We use 256×256 inputs for the live preview.


In [ ]:
from modnet_weights import load_checkpoint

model = MODNet(backbone_pretrained=False).eval()
checkpoint = await load_checkpoint()
model.load_state_dict({name.removeprefix('module.'): value
                       for name, value in checkpoint.items()}, strict=True)
del checkpoint
for parameter in model.parameters():
    parameter.requires_grad_(False)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

def prepare_image(bgr):
    rgb = cv2.cvtColor(cv2.resize(bgr, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR),
                       cv2.COLOR_BGR2RGB)
    normalized = rgb.astype(np.float32) / 127.5 - 1
    return torch.from_numpy(normalized.transpose(2, 0, 1).copy()).unsqueeze(0)

def predict_matte(image):
    return model(image, True)[2]

sample = cv2.imread('astronaut.png')
assert sample is not None
example = prepare_image(sample)
with torch.inference_mode():
    expected = predict_matte(example)
assert torch.isfinite(expected).all()
print('Eager reference ready:', tuple(expected.shape))


## 2. Compile and check the model

Compile on the selected device with `torch.compile`. The first call includes
compilation; check its prediction against eager PyTorch before opening the camera.


In [ ]:
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 1), name='webgpu')
    backend = gpu.backend
else:
    gpu = None
    backend = 'turbine_cpu'

model = model.to(DEVICE)
example_device = example.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
compiled_model = torch.compile(predict_matte, backend=backend,
                               fullgraph=True, dynamic=False)

started = time.perf_counter()
with torch.inference_mode():
    alpha = compiled_model(example_device)
actual = await gpu.to_cpu(alpha) if gpu is not None else alpha.detach()
cold_seconds = time.perf_counter() - started
torch.testing.assert_close(actual, expected, rtol=3e-3, atol=8e-4)
report = dict(passed=True, device=DEVICE, input_size=SIZE, opencv=cv2.__version__,
              cold_first_call_seconds=cold_seconds,
              max_abs_error=float((actual - expected).abs().max()))
print(json.dumps(report, indent=2))


## 3. Process each frame

Prepare the image with OpenCV, predict the foreground mask and apply the effect.
The panel waits for each result before requesting another frame.


In [ ]:
async def process_frame(jpeg, effect):
    started = time.perf_counter()
    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None or max(frame.shape[:2]) > 1024:
        raise ValueError('Expected a valid camera image no larger than 1024 pixels.')
    prepared = prepare_image(frame)
    prepared_at = time.perf_counter()
    device_input = prepared.to(DEVICE)
    with torch.inference_mode():
        prediction = compiled_model(device_input)
    alpha_cpu = await gpu.to_cpu(prediction) if gpu is not None else prediction.detach()
    matte = alpha_cpu[0, 0].numpy()
    if not np.isfinite(matte).all() or matte.min() < -1e-4 or matte.max() > 1.0001:
        raise ValueError('The model returned an invalid foreground matte.')
    model_at = time.perf_counter()
    matte = cv2.resize(np.clip(matte, 0, 1), (frame.shape[1], frame.shape[0]),
                       interpolation=cv2.INTER_LINEAR)[..., None]

    if effect == 'cutout':
        result = np.concatenate((frame, np.rint(matte * 255).astype(np.uint8)), axis=2)
    elif effect == 'matte':
        result = np.rint(matte[..., 0] * 255).astype(np.uint8)
    else:
        if effect == 'blur':
            background = cv2.GaussianBlur(frame, (0, 0), 12)
        elif effect == 'color':
            background = np.full_like(frame, (166, 140, 20))
        elif effect == 'colorpop':
            background = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),
                                       cv2.COLOR_GRAY2BGR)
        elif effect == 'pixelate':
            small = cv2.resize(frame, (20, 15), interpolation=cv2.INTER_AREA)
            background = cv2.resize(small, (frame.shape[1], frame.shape[0]),
                                     interpolation=cv2.INTER_NEAREST)
        else:
            raise ValueError(f'Unknown effect: {effect}')
        result = np.rint(frame * matte + background * (1 - matte)).clip(0, 255).astype(np.uint8)
    effect_at = time.perf_counter()
    ok, encoded = cv2.imencode('.png', result, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    if not ok:
        raise RuntimeError('OpenCV could not encode the result.')
    png = base64.b64encode(encoded).decode()
    finished = time.perf_counter()
    timings = dict(prepare_ms=1000 * (prepared_at - started),
                   model_ms=1000 * (model_at - prepared_at),
                   effect_ms=1000 * (effect_at - model_at),
                   encode_ms=1000 * (finished - effect_at),
                   python_total_ms=1000 * (finished - started))
    return dict(png=png, effect=effect, timings=timings)


## 4. Try your camera

Click **Start camera** and allow access, or try the included photo first.
Drag the before/after slider, switch to side-by-side view, and try the effects.
**Stop camera** releases the camera. Save the latest result as a PNG;
background removal preserves transparency.


In [ ]:
from live_vision import LiveVision

if 'camera' in globals():
    await camera.close()
camera = LiveVision(process_frame, sample='astronaut.png', size=SIZE, device=DEVICE)


When finished, click **Stop camera**. To close the panel from Python, run
`await camera.close()` in a new cell. Re-running the panel cell closes its previous instance.

The displayed frame rate includes the complete image-processing update.
Lighting, motion and fine hair can affect the mask.

Model and weights: [MODNet](https://github.com/ZHKKKe/MODNet), Apache-2.0;
see `LICENSE-MODNet` and `sources.json`. Sample portrait: NASA, Eileen Collins, public domain.
